# Vignette 07: Immunotherapy Response Prediction
## A Clinical Case Study with the LORIS Cohort

This vignette demonstrates **mlsurv** on a real-world immunotherapy dataset from Chang et al., *Nature Cancer* 2024. We:

1. Train survival models on the Chowell pan-cancer immunotherapy cohort
2. Benchmark against three published reference scores: **LORIS**, **TMB**, and **PD-L1**
3. Validate on 7 external cohorts (MSK1, MSK2, 3 NSCLC cohorts, 2 pan-cancer cohorts)
4. Examine per-cancer-type performance via subpopulation analysis
5. Generate a patient-level explanation with SHAP
6. Predict outcomes for new patients with bootstrap confidence intervals
7. Explore feature interactions (H-statistic and SHAP interaction values)

**Data**: 10 cohorts from the [LORIS GitHub repository](https://github.com/rootchang/LORIS), extracted from `AllData.xlsx` into per-cohort CSVs in `data/loris/`.

**Outcomes**: Both Overall Survival (OS) and Progression-Free Survival (PFS) are available. Set `OUTCOME` below to choose. PFS has more events (84% vs 64%) and is the more clinically relevant ICB endpoint.

*Runtime: ~35 minutes on a modern laptop.*

In [ ]:
import os, sys, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from mlsurv import SurvivalLearner
from mlsurv.utils import to_structured_array
from mlsurv.guidance import recommend_models
from mlsurv._warnings import SuggestionWarning

# ── Configuration ─────────────────────────────────────────────────────────────
OUTCOME = 'OS'  # 'OS' or 'PFS'
TIME_COL = f'{OUTCOME}_Months'
EVENT_COL = f'{OUTCOME}_Event'

# loris_score_calculator.py lives in ../scripts (run this notebook from examples/)
sys.path.insert(0, '../scripts')
from loris_score_calculator import calculate_loris

print(f'Outcome: {OUTCOME} ({TIME_COL}, {EVENT_COL})')

## 1. Data Loading and Exploration

The Chowell cohort is pre-split into training (N~964) and test (N~515) sets, matching the original LORIS paper methodology.

In [ ]:
# Load pre-split Chowell cohort
DATA_DIR = Path('data/loris')

df_train = pd.read_csv(DATA_DIR / 'Chowell_train.csv')
df_test = pd.read_csv(DATA_DIR / 'Chowell_test.csv')

# Print dataset info
for label, df in [('Train', df_train), ('Test', df_test)]:
    n_ev = int(df[EVENT_COL].sum())
    print(f'{label}: {len(df)} patients, {n_ev} {OUTCOME} events ({df[EVENT_COL].mean():.0%}), '
          f'median {OUTCOME} {df[TIME_COL].median():.1f} months')

In [ ]:
# Print cancer type distribution table
ct_train = df_train['CancerType'].value_counts().rename('Train')
ct_test = df_test['CancerType'].value_counts().rename('Test')
ct_table = pd.concat([ct_train, ct_test], axis=1).fillna(0).astype(int)
ct_table['Total'] = ct_table.sum(axis=1)
ct_table = ct_table.sort_values('Total', ascending=False)
print(ct_table.to_string())
print(f'\n{len(ct_table)} cancer types')

In [ ]:
# Feature engineering: one-hot encode CancerType
CLINICAL_FEATURES = ['TMB', 'Age', 'Systemic_therapy_history', 'Albumin', 'NLR']

def prepare_features(df, train_columns=None):
    """Build feature matrix with clinical + one-hot cancer type columns."""
    X = df[CLINICAL_FEATURES].copy()
    ct_dummies = pd.get_dummies(df['CancerType'], prefix='CancerType')
    X = pd.concat([X, ct_dummies], axis=1)
    if train_columns is not None:
        X = X.reindex(columns=train_columns, fill_value=0)
    return X

X_train = prepare_features(df_train)
X_test = prepare_features(df_test, train_columns=X_train.columns)

y_train = to_structured_array(df_train[TIME_COL], df_train[EVENT_COL])
y_test = to_structured_array(df_test[TIME_COL], df_test[EVENT_COL])

# Detect one-hot encoded groups for grouped permutation importance
from mlsurv.preprocessing import detect_categorical_groups
cat_groups = detect_categorical_groups(X_train)

# Print feature info
print(f'Features: {X_train.shape[1]} ({len(CLINICAL_FEATURES)} clinical + '
      f'{X_train.shape[1] - len(CLINICAL_FEATURES)} cancer type dummies)')
print(f'Categorical groups: {cat_groups}')
print(f'Events: {int(y_train["event"].sum())} train, {int(y_test["event"].sum())} test')

In [ ]:
# PD-L1 availability
pdl1_train = df_train['PDL1_TPS'].notna().mean()
pdl1_test = df_test['PDL1_TPS'].notna().mean()
print(f'PD-L1 availability: train {pdl1_train:.0%}, test {pdl1_test:.0%}')
print(f'  -> PD-L1 benchmark will evaluate on {df_test["PDL1_TPS"].notna().sum()} '
      f'of {len(df_test)} test patients')

## 2. Model Training

We use the pre-split train/test sets from the LORIS paper. Six models are compared: CoxPH, CoxNet, Random Survival Forest, XGBCox (Cox objective), XGBoost-AFT (Accelerated Failure Time objective), and FSSVM (Fast Survival SVM, the only `Recommended` model per `recommend_models()`). Deep learning models are skipped for runtime efficiency.

In [ ]:
# Derive cancer type labels for subpopulation analysis
cancer_train = df_train['CancerType'].values
cancer_test = df_test['CancerType'].values

learner = SurvivalLearner(
    X_train=X_train, X_test=X_test,
    y_train=y_train, y_test=y_test,
)
learner.setup(
    categorical_groups=cat_groups,
    subpopulations_train={'Cancer Type': cancer_train},
    subpopulations_test={'Cancer Type': cancer_test},
    stratify_subpopulations=True,
    cv_folds=5,
    random_state=42,
    n_jobs=-1,
)
print('Setup complete.')

In [ ]:
# Train and evaluate models (with feature interactions)
results = learner.run(
    models=['coxph', 'coxnet', 'rsf', 'xgbcox', 'xgbaft', 'fssvm'],
    n_trials=15,
    n_bootstrap=100,
    eval_times=[1, 2, 3, 6, 9, 12, 18],  # clinically meaningful ICB landmarks
    include_interactions=True,
    interaction_top_k=5,
    show_eval_progress=False,
)

In [ ]:
# KM survival curves for best model
learner.plot_survival(learner.best_model.name, n_groups=3)

## 3. Benchmark: LORIS, TMB, PD-L1

We compare our trained models against three published reference biomarkers:

- **LORIS** (Chang et al., 2024): 6-variable logistic regression with cancer-type calibration. Computed using published coefficients.
- **TMB** (Tumor Mutational Burden): raw value, higher = better prognosis under immunotherapy.
- **PD-L1** (Tumor Proportion Score): available for ~36% of the test set. mlsurv's benchmark() handles the NaN subsetting automatically.

In [ ]:
# Compute reference scores on test set
loris_scores = calculate_loris(df_test).values
tmb_scores = df_test['TMB'].values
pdl1_scores = df_test['PDL1_TPS'].values  # ~64% NaN — handled by benchmark

print(f'LORIS: mean={np.nanmean(loris_scores):.3f}, '
      f'range=[{np.nanmin(loris_scores):.3f}, {np.nanmax(loris_scores):.3f}]')
print(f'TMB:   mean={np.nanmean(tmb_scores):.1f}, '
      f'range=[{np.nanmin(tmb_scores):.1f}, {np.nanmax(tmb_scores):.1f}]')
print(f'PD-L1: {np.isnan(pdl1_scores).sum()} NaN of {len(pdl1_scores)} '
      f'({np.isnan(pdl1_scores).mean():.0%})')

In [ ]:
# Run benchmark
bench = learner.benchmark(
    references={
        'LORIS': loris_scores,
        'TMB': tmb_scores,
        'PD-L1': pdl1_scores,
    },
    higher_is_worse=False,
    n_bootstrap=100,
    random_state=42,
    verbose=False,
)
# Concordance table
if bench.evaluation and bench.evaluation.concordance is not None:
    print(bench.evaluation.concordance[['model_type', 'c_index_test', 'c_index_ipcw_test']].to_string(index=False))

In [ ]:
# Benchmark metric comparison
learner.plot_metric(style='forest')

In [ ]:
# Feature importance
learner.plot_importance(n=15)

## 4. External Validation

We validate the trained models on independent cohorts:

| Cohort | N | Type |
|--------|---|------|
| MSK1 | ~453 | Pan-cancer |
| MSK2 | ~104 | CNS + unknown primary |
| Shim_NSCLC | ~198 | NSCLC |
| Vanguri_NSCLC | ~246 | NSCLC (PFS only) |
| Ravi_NSCLC | ~309 | NSCLC |
| Pradat_panCancer | ~57 | Pan-cancer |
| Kato_panCancer | ~35 | Pan-cancer |

Note: LORIS included Vanguri_NSCLC, but for OS it has no censored patients — all survivors lack follow-up times, so after dropping rows with missing OS columns the cohort is all events. That makes Harrell's C and the IPCW metrics unreliable, so Vanguri_NSCLC is excluded from OS external validation (6 cohorts for OS, 7 for PFS). It is retained for the PFS endpoint, where its censoring is intact (246 patients, 209 events, 37 censored).

MSK_nonICB (N~841) is a negative control — patients treated without immunotherapy.

In [ ]:
# Validate on external cohorts.
#
# Each cohort gets head-to-head metrics for our models AND the published
# reference biomarkers (LORIS, TMB, PD-L1) scored on that same cohort,
# via the references= parameter on validate(). Reference
# rows are appended to the cohort's concordance table — no manual scoring
# step required.
#
# Vanguri_NSCLC is excluded from OS validation: after dropping rows with missing
# OS_Months/OS_Event it has 150 patients, all 150 events, 0 censored (the ~96
# dropped rows are survivors lacking follow-up times — informative missingness).
# Conditioning on death this way biases both the IPCW-family metrics (censoring
# weights collapse to 1) and Harrell's C, so it is not a valid OS external cohort.
# It is valid for PFS (246 patients, 209 events, 37 censored) and is kept there.
VALIDATION_COHORTS = [
    'MSK1', 'MSK2', 'Shim_NSCLC', 'Ravi_NSCLC',
    'Pradat_panCancer', 'Kato_panCancer',
]
if OUTCOME == 'PFS':
    VALIDATION_COHORTS.insert(3, 'Vanguri_NSCLC')  # PFS-only; see note above
EVAL_TIMES = [1, 2, 3, 6, 9, 12, 18]  # matches run() eval_times


def _build_cohort_references(df_ext):
    """Compute LORIS / TMB / PD-L1 for an external cohort when columns exist."""
    refs = {}
    hiw = {}
    try:
        loris = calculate_loris(df_ext).values
        refs['LORIS'] = loris
        hiw['LORIS'] = False  # higher LORIS = better outcome
    except (KeyError, ValueError):
        pass  # missing one of the 6 LORIS variables for this cohort
    if 'TMB' in df_ext.columns:
        refs['TMB'] = df_ext['TMB'].values
        hiw['TMB'] = False  # higher TMB = better prognosis
    if 'PDL1_TPS' in df_ext.columns:
        refs['PD-L1'] = df_ext['PDL1_TPS'].values
        hiw['PD-L1'] = False  # higher PD-L1 = better prognosis
    return refs, hiw


for cohort_name in VALIDATION_COHORTS:
    df_ext = pd.read_csv(DATA_DIR / f'{cohort_name}.csv')
    if TIME_COL not in df_ext.columns or EVENT_COL not in df_ext.columns:
        print(f'{cohort_name:20s}  skipped (no {OUTCOME} columns)')
        continue
    df_ext = df_ext.dropna(subset=[TIME_COL, EVENT_COL])
    if len(df_ext) < 10:
        print(f'{cohort_name:20s}  skipped ({len(df_ext)} valid rows)')
        continue
    X_ext = prepare_features(df_ext, train_columns=X_train.columns)
    y_ext = to_structured_array(df_ext[TIME_COL], df_ext[EVENT_COL])
    cancer_ext = df_ext['CancerType'].values

    refs, hiw = _build_cohort_references(df_ext)

    learner.validate(
        X_ext, y_ext,
        name=cohort_name,
        subpopulations={'Cancer Type': cancer_ext},
        references=refs if refs else None,
        higher_is_worse=hiw if hiw else True,
        eval_times=EVAL_TIMES,
        verbose=False,
        bootstrap=False,
        show_progress=False,
    )
    n_events = int(y_ext['event'].sum())
    ref_note = ', '.join(refs.keys()) if refs else 'no refs'
    print(f'{cohort_name:20s}  N={len(df_ext):4d}  events={n_events:3d}  '
          f'types={df_ext["CancerType"].nunique()}  [{ref_note}]')

### Negative Control: MSK Non-ICB Cohort

The MSK_nonICB cohort contains patients who did **not** receive immunotherapy. A well-calibrated immunotherapy-specific score should show reduced discriminatory performance on this cohort, since the survival signal is prognostic rather than predictive of ICB response.

In [ ]:
# Negative control: non-ICB cohort
df_nonicb = pd.read_csv(DATA_DIR / 'MSK_nonICB.csv')
if TIME_COL in df_nonicb.columns and EVENT_COL in df_nonicb.columns:
    df_nonicb = df_nonicb.dropna(subset=[TIME_COL, EVENT_COL])
    X_nonicb = prepare_features(df_nonicb, train_columns=X_train.columns)
    y_nonicb = to_structured_array(df_nonicb[TIME_COL], df_nonicb[EVENT_COL])
    refs, hiw = _build_cohort_references(df_nonicb)
    learner.validate(
        X_nonicb, y_nonicb,
        name='MSK_nonICB (neg ctrl)',
        subpopulations={'Cancer Type': df_nonicb['CancerType'].values},
        references=refs if refs else None,
        higher_is_worse=hiw if hiw else True,
        eval_times=EVAL_TIMES,
        verbose=False,
        bootstrap=False,
        show_progress=False,
    )
    ref_note = ', '.join(refs.keys()) if refs else 'no refs'
    print(f'MSK_nonICB: N={len(df_nonicb)}, events={int(y_nonicb["event"].sum())}  [{ref_note}]')
else:
    print(f'MSK_nonICB: skipped (no {OUTCOME} columns — OS only for this cohort)')

## 5. Patient-Level Explanation

SHAP waterfall plot for an individual patient, showing which features drive their predicted risk.

In [ ]:
# Explain a high-risk test patient using the best model. `patient='high_risk'`
# is the same 90th-percentile shorthand used by patient_report.
explanation = learner.explain(patient='high_risk', model=learner.best_model.name)
explanation.figure

## 6. Patient-Level Reports

`learner.patient_report()` produces a standalone HTML report for a single patient that bundles:

- the **individualized survival curve** (with bootstrap CI band),
- **milestone P(survive > t)** at clinician-specified times,
- a **risk-score histogram** placing the patient in the test cohort,
- a **KM overlay** comparing the patient to risk groups,
- a **SHAP waterfall** explaining the prediction, and
- a **k-nearest-neighbors table** showing the most-similar training patients.

We generate two reports — one **low-risk** patient (10th percentile of model risk score on the test set) and one **high-risk** patient (90th percentile) — using the best CV-Harrell-C model.

In [ ]:
# Generate reports for a low- and a high-risk test patient using the best
# CV-Harrell-C model. `patient='low_risk'`/`'high_risk'` is shorthand for the
# 10th/90th-percentile of the model's risk score on the passed X.
best_name = learner.best_model.name
os.makedirs('output/vig7', exist_ok=True)

for label in ('low_risk', 'high_risk'):
    out_path = f'output/vig7/patient_{label}_{OUTCOME}.html'
    learner.patient_report(
        X_test, patient=label, model=best_name,
        output_path=out_path,
        times=EVAL_TIMES, time_unit='months',
        ci=True, n_bootstrap=200, random_state=42,
    )
    print(f'{label:9s} ({best_name}) -> {out_path}')

## 7. Feature Interactions

Feature interactions reveal which pairs of variables jointly influence survival beyond their individual effects. We use Friedman's H-statistic (model-agnostic, computed during `run()`) on the Random Survival Forest, then compare with SHAP interaction values from TreeExplainer.

Note: H-statistic is only computed for non-linear models — linear models (CoxPH, CoxNet) have no interaction structure to detect.

In [ ]:
# H-statistic interaction heatmap (RSF)
learner.plot_interactions(model='rsf', n=10)

In [ ]:
# 2D PDP for the top interacting pair (RSF)
interactions_df = learner.plot_interactions(model='rsf', return_data=True)
top_pair = interactions_df.iloc[0]
f1, f2 = top_pair['feature_1'], top_pair['feature_2']
print(f"Top interaction: {f1} \u00d7 {f2} (H = {top_pair['h_statistic']:.3f})")

learner.plot_interaction_pdp(f1, f2, model='rsf')

### SHAP Interaction Values (Tree-Based Models)

For tree-based models (RSF, GBSA, XGBoost), SHAP interaction values from TreeExplainer provide a complementary view. Unlike the H-statistic (which is model-agnostic), SHAP interactions decompose per-sample predictions into pairwise contributions.

In [ ]:
# Save H-statistic rankings for RSF before recomputing as SHAP
hstat_rsf = learner.plot_interactions(model='rsf', return_data=True)

# Compute SHAP interaction values for RSF (tree-based)
learner.interactions(model='rsf', method='shap', top_k=5, show_progress=False)

# Get SHAP interaction rankings
shap_rsf = learner.plot_interactions(model='rsf', return_data=True)

# Side-by-side comparison: H-statistic vs SHAP interactions
comparison = hstat_rsf[['feature_1', 'feature_2', 'h_statistic']].merge(
    shap_rsf[['feature_1', 'feature_2', 'shap_interaction']],
    on=['feature_1', 'feature_2'],
    how='outer',
).sort_values('h_statistic', ascending=False, na_position='last')
comparison.index = range(1, len(comparison) + 1)
comparison.index.name = 'Rank'
print(comparison.to_string())

## 8. Save the Learner

Persist the complete learner state to disk so downstream manuscript figures — e.g. the raincloud of bootstrap test-C distributions across all models and the LORIS / TMB / PD-L1 benchmarks — can be regenerated via `SurvivalLearner.load_learner(...)` without re-running the full analysis. The filename is suffixed with `OUTCOME` (`learner_OS.pkl` / `learner_PFS.pkl`) so the two endpoints write to separate files.

In [ ]:
# Save the full learner (data, models, evaluation, bootstrap CIs, benchmark,
# external validation) so manuscript figures can be reproduced without re-running
# the ~35 min analysis. The filename tracks OUTCOME so the OS and PFS runs
# write to separate files and never clobber each other.
learner_path = f'output/vig7/learner_{OUTCOME}.pkl'
learner.save_learner(learner_path)
print(f'Saved learner -> {learner_path}')

## 9. Report and Export

Generate an interactive HTML report covering model development, benchmarking, external validation, and feature importance. Also export manuscript-ready tables to Excel.

In [ ]:
# Feature groups for report visualizations
# Map both raw dummy columns (used by correlation heatmap) and grouped names
# (used by importance plots) to their thematic group.
FEATURE_GROUPS = {}
for col in X_train.columns:
    if col.startswith('CancerType_'):
        FEATURE_GROUPS[col] = 'cancer_type'
    elif col == 'TMB':
        FEATURE_GROUPS[col] = 'molecular'
    elif col in ('Albumin', 'NLR'):
        FEATURE_GROUPS[col] = 'blood'
    else:
        FEATURE_GROUPS[col] = 'clinical'
# Also map the grouped name used in importance output
for group_name in cat_groups:
    FEATURE_GROUPS[group_name] = 'cancer_type'

GROUP_COLORS = {
    'molecular': '#2ca02c',
    'blood': '#d62728',
    'clinical': '#1f77b4',
    'cancer_type': '#ff7f0e',
}

print(f'Feature groups: {dict(pd.Series(FEATURE_GROUPS).value_counts())}')

In [ ]:
# Generate report
os.makedirs('output/vig7', exist_ok=True)

learner.generate_report(
    f'output/vig7/report_{OUTCOME}.html',
    title=f'Vignette 07: Immunotherapy Response Prediction ({OUTCOME})',
    feature_groups=FEATURE_GROUPS,
    group_colors=GROUP_COLORS,
)
print('Report generated.')

In [ ]:
# Export results + manuscript-ready tables
learner.export_results(f'output/vig7/results_{OUTCOME}.xlsx')

# Manuscript-ready tables: benchmark concordance + subpopulation pivot + interactions
sg_result = learner.subpopulation('Cancer Type').result

with pd.ExcelWriter(f'output/vig7/manuscript_tables_{OUTCOME}.xlsx') as w:
    # Benchmark concordance (models + references)
    if bench.evaluation and bench.evaluation.concordance is not None:
        conc = bench.evaluation.concordance.copy()
        if bench.sample_sizes:
            conc['n_evaluated'] = conc['model_type'].map(bench.sample_sizes)
        conc.to_excel(w, sheet_name='benchmark', index=False)

    # Per-cancer-type subpopulation metrics and pivot table
    sg_result.metrics.to_excel(w, sheet_name='subpopulation_metrics', index=False)
    sg_result.summary.to_excel(w, sheet_name='subpopulation_summary', index=False)

    harrell = sg_result.metrics[sg_result.metrics['metric'] == 'harrell']
    pivot = harrell.pivot(index='subpopulation', columns='model_type', values='value')
    summary = sg_result.summary.set_index('subpopulation')[['n_samples', 'n_events']]
    pivot = summary.join(pivot).sort_values('n_samples', ascending=False)
    pivot.to_excel(w, sheet_name='subpopulation_pivot')

    # Feature interaction strengths (H-statistic + SHAP for RSF)
    all_interactions = learner.plot_interactions(return_data=True)
    if all_interactions is not None and len(all_interactions) > 0:
        all_interactions.to_excel(w, sheet_name='interactions', index=False)

print(f'Exported: manuscript_tables_{OUTCOME}.xlsx')

## Summary

This vignette demonstrated a complete immunotherapy case study with mlsurv:

- **Training**: 6 survival models on the Chowell pan-cancer cohort (16 cancer types)
- **Benchmarking**: Against LORIS (published score), TMB, and PD-L1 (with NaN handling)
- **External validation**: 6 cohorts for OS / 7 for PFS, plus a negative control (Vanguri_NSCLC is PFS-only — see §4)
- **Subpopulation analysis**: Per-cancer-type performance with bootstrap CIs
- **Explainability**: SHAP waterfall for individual patients
- **Individual prediction**: Risk scores, survival probabilities, and median survival with bootstrap CIs
- **Feature interactions**: H-statistic heatmap and 2D PDP surfaces
- **SHAP interactions**: TreeExplainer-based interaction values for tree models

**Exported**: `manuscript_tables_{OS,PFS}.xlsx` contains benchmark concordance, subpopulation metrics, pivot table, and interaction strengths for manuscript figures. Run with `OUTCOME='PFS'` for PFS results.

## References

- Chang TG, Cao Y, Sfreddo HJ, et al. (2024). LORIS robustly predicts patient outcomes with immune checkpoint blockade therapy using common clinical, pathologic and genomic features. *Nature Cancer*, 5(8):1158–1175. https://doi.org/10.1038/s43018-024-00772-7
- Chowell D, Morris LGT, Grigg CM, et al. (2018). Patient HLA class I genotype influences cancer response to checkpoint blockade immunotherapy. *Science*, 359(6375):582–587. https://doi.org/10.1126/science.aao4572
- Friedman JH, Popescu BE (2008). Predictive learning via rule ensembles. *The Annals of Applied Statistics*, 2(3):916–954. https://doi.org/10.1214/07-AOAS148
- Harrell FE, Califf RM, Pryor DB, Lee KL, Rosati RA (1982). Evaluating the yield of medical tests. *JAMA*, 247(18):2543–2546. https://doi.org/10.1001/jama.1982.03320430047030
- Lundberg SM, Lee S-I (2017). A unified approach to interpreting model predictions. In *Advances in Neural Information Processing Systems*, 30:4766–4777. arXiv:1705.07874. https://arxiv.org/abs/1705.07874
- Samstein RM, Lee CH, Shoushtari AN, et al. (2019). Tumor mutational load predicts survival after immunotherapy across multiple cancer types. *Nature Genetics*, 51(2):202–206. https://doi.org/10.1038/s41588-018-0312-8